In [ ]:
import copy
import random
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold, train_test_split

# Splits:  all data -> test set (SPLIT 1, main) + train+validation set
#          train+validation -> CV folds (SPLIT 2, cross_validate)
#          any training data -> fit set + early-stopping set (SPLIT 3, train_model)

DATASET_PATH = "Datasets/kryptonite-20"

N_FOLDS = 5
SEEDS = [0, 1, 2]
SPLIT_SEED = 0

BATCH_SIZE = 256
MAX_EPOCHS = 150
PATIENCE = 20               # stop after this many epochs in a row with no improvement in early-stopping loss
EARLY_STOPPING_SIZE = 0.20  # fraction of the training data used to decide when to stop
TEST_SIZE = 0.20            # final test set

LEARNING_RATES = [3e-3, 1e-3, 3e-4]
WEIGHT_DECAYS = [0.0, 1e-2]
DROPOUT_RATES = [0.0, 0.1]

ARCHITECTURES = [(32,),
                 (32, 32),
                 (64, 64, 64)]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

X = np.load(f"{DATASET_PATH}-X.npy").astype(np.float32)
y = np.load(f"{DATASET_PATH}-y.npy").reshape(-1).astype(np.float32)


def to_tensor(array):
    return torch.tensor(array, device=DEVICE)


def build_mlp(input_dim, hidden_sizes, dropout):
    layers, dim = [], input_dim
    for size in hidden_sizes:
        layers += [nn.Linear(dim, size), nn.ReLU(), nn.Dropout(dropout)]
        dim = size
    layers.append(nn.Linear(dim, 1))
    return nn.Sequential(*layers)


def train_model(hidden_sizes, dropout, lr, weight_decay, seed, x_train, y_train):
    """Train one MLP on x_train/y_train and return it with its best early-stopping weights."""
    # SPLIT 3: every run splits its own training data into a fit set (weights are
    # learned from it) and an early-stopping set (only used to decide when to stop).
    x_fit, x_stop, y_fit, y_stop = train_test_split(
        x_train, y_train, test_size=EARLY_STOPPING_SIZE,
        stratify=y_train, random_state=SPLIT_SEED
    )
    x_fit, y_fit, x_stop, y_stop = map(
        to_tensor, (x_fit, y_fit, x_stop, y_stop))

    # Seed BEFORE building the model so the seed controls weight init.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = build_mlp(x_fit.shape[1], hidden_sizes, dropout).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    generator = torch.Generator().manual_seed(seed)

    best_loss, best_state, bad_epochs = float(
        "inf"), copy.deepcopy(model.state_dict()), 0

    for _ in range(MAX_EPOCHS):
        model.train()
        order = torch.randperm(len(x_fit), generator=generator).to(DEVICE)

        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            optimizer.zero_grad()
            loss_fn(model(x_fit[idx]).squeeze(1), y_fit[idx]).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            stop_loss = loss_fn(model(x_stop).squeeze(1), y_stop).item()

        if stop_loss < best_loss:
            best_loss, best_state, bad_epochs = stop_loss, copy.deepcopy(
                model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model


def compute_accuracy(model, x, y_true):
    """Fraction of examples the model classifies correctly."""
    model.eval()
    with torch.no_grad():
        predictions = (model(to_tensor(x)).squeeze(1) > 0).float()
    return (predictions == to_tensor(y_true)).float().mean().item()


def mean_accuracy_over_seeds(hidden_sizes, dropout, lr, weight_decay,
                             x_train, y_train, x_eval, y_eval):
    """Train once per seed on the training data; return mean accuracy on the evaluation data."""
    accuracies = []
    for seed in SEEDS:
        model = train_model(hidden_sizes, dropout, lr,
                            weight_decay, seed, x_train, y_train)
        accuracies.append(compute_accuracy(model, x_eval, y_eval))
    return np.mean(accuracies)


def cross_validate(hidden_sizes, dropout, lr, weight_decay, x_trainval, y_trainval):
    """Return the accuracy of one config on each validation fold."""
    # SPLIT 2: each fold takes a turn as the validation fold; the rest are training folds.
    folds = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
    fold_accuracies = []
    for train_idx, val_idx in folds.split(x_trainval, y_trainval):
        fold_accuracies.append(mean_accuracy_over_seeds(
            hidden_sizes, dropout, lr, weight_decay,
            x_train=x_trainval[train_idx], y_train=y_trainval[train_idx],
            x_eval=x_trainval[val_idx], y_eval=y_trainval[val_idx],
        ))
    return fold_accuracies


def cross_validate_all_configs(x_trainval, y_trainval):
    """Score every architecture x hyperparameter combination by cross-validation."""
    results = []
    for hidden_sizes in ARCHITECTURES:
        arch_name = "-".join(map(str, hidden_sizes)) 
        n_params = sum(p.numel() for p in build_mlp(
            X.shape[1], hidden_sizes, 0.0).parameters())

        for lr, wd, dropout in product(LEARNING_RATES, WEIGHT_DECAYS, DROPOUT_RATES):
            fold_accuracies = cross_validate(
                hidden_sizes, dropout, lr, wd, x_trainval, y_trainval)
            results.append({
                "architecture": arch_name,
                "hidden_sizes": hidden_sizes,
                "parameters": n_params,
                "learning_rate": lr,
                "weight_decay": wd,
                "dropout_rate": dropout,
                "cv_accuracy": np.mean(fold_accuracies),
                "cv_std": np.std(fold_accuracies, ddof=1),
            })
            print(
                f"{arch_name} lr={lr} wd={wd} dropout={dropout} -> {np.mean(fold_accuracies):.4f}")
    return pd.DataFrame(results)


def select_best_config_per_architecture(results):
    """Keep the highest-CV-accuracy row for each architecture."""
    return (
        results.sort_values("cv_accuracy", ascending=False)
        .groupby("architecture").head(1)
        .reset_index(drop=True)
    )


def test_accuracy_of_each_winner(best, x_trainval, y_trainval, x_test, y_test):
    """Train each winning config on all train+validation data; return its test-set accuracy."""
    return [
        mean_accuracy_over_seeds(
            row["hidden_sizes"], row["dropout_rate"],
            row["learning_rate"], row["weight_decay"],
            x_train=x_trainval, y_train=y_trainval, x_eval=x_test, y_eval=y_test,
        )
        for _, row in best.iterrows()
    ]


def main():
    # SPLIT 1: the test set is set aside first and only used for the final scores,
    # so final test accuracy is unbiased 
    x_trainval, x_test, y_trainval, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=SPLIT_SEED
    )

    results = cross_validate_all_configs(x_trainval, y_trainval)

    best = select_best_config_per_architecture(results)
    best["test_accuracy"] = test_accuracy_of_each_winner(
        best, x_trainval, y_trainval, x_test, y_test)

    print(best.to_string(index=False))
    best.to_csv("k20_architecture_comparison.csv", index=False)


if __name__ == "__main__":
    main()

32 lr=0.003 wd=0.0 dropout=0.0 -> 0.7394
64-64-64 lr=0.003 wd=0.0 dropout=0.0 -> 0.8331
architecture hidden_sizes  parameters  learning_rate  weight_decay  dropout_rate  cv_accuracy   cv_std  test_accuracy
    64-64-64 (64, 64, 64)        9729          0.003           0.0           0.0     0.833056 0.026518       0.904583
          32        (32,)         705          0.003           0.0           0.0     0.739410 0.012244       0.760139
